# Fase 3 -- Camada Gold: Análise e Perguntas de Negócio

Este notebook consome a camada **Silver** (dados limpos e tipados) para responder
às perguntas de negócio do desafio, e constrói a camada **Gold** (uma tabela
agregada + uma view, feitas com `JOIN` + `GROUP BY`) que fica salva no
PostgreSQL para consultas futuras.

Estrutura deste notebook:

1. Conexão com o banco
2. Perguntas 1 a 3 -- respondidas direto sobre a Silver (SQL + tabela + gráfico)
3. Construção da camada Gold (`gold_orgao_resumo`, como tabela e como view)
4. Perguntas 4 a 6 -- respondidas com apoio da Gold/Silver (SQL + tabela + gráfico)
5. Conclusões e insights


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

import banco

pd.set_option("display.max_colwidth", 80)

## 1. Conexão com o banco

In [ ]:
conexao = banco.conectar()
print("Conectado ao PostgreSQL com sucesso.")

## 2. Perguntas de negócio 1 a 3 (camada Silver)

In [ ]:
sql_q1 = """
    SELECT nome_orgao_superior,
           SUM(valor_total) AS custo_total
    FROM silver_viagem
    GROUP BY nome_orgao_superior
    ORDER BY custo_total DESC
    LIMIT 5;
"""
df_q1 = pd.read_sql(sql_q1, conexao)
df_q1

In [ ]:
plt.figure(figsize=(8, 5))
plt.barh(df_q1["nome_orgao_superior"], df_q1["custo_total"], color="#2E86AB")
plt.xlabel("Custo total (R$)")
plt.title("Top 5 órgãos com maior custo total em viagens")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


### Pergunta 2 -- Os 3 destinos com maior custo médio por viagem?

In [ ]:
sql_q2 = """
    SELECT destinos,
           COUNT(*) AS qtd_viagens,
           ROUND(AVG(valor_total), 2) AS custo_medio
    FROM silver_viagem
    WHERE destinos IS NOT NULL
    GROUP BY destinos
    HAVING COUNT(*) >= 5
    ORDER BY custo_medio DESC
    LIMIT 3;
"""
df_q2 = pd.read_sql(sql_q2, conexao)
df_q2

In [ ]:
plt.figure(figsize=(8, 4))
plt.barh(df_q2["destinos"], df_q2["custo_medio"], color="#A23B72")
plt.xlabel("Custo médio por viagem (R$)")
plt.title("Top 3 destinos com maior custo médio por viagem")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

### Pergunta 3 -- A viagem de maior duração e seu custo total?

In [ ]:
sql_q3 = """
    SELECT id_viagem, nome_orgao_superior, nome_viajante, destinos,
           data_inicio, data_fim, duracao_dias, valor_total
    FROM silver_viagem
    ORDER BY duracao_dias DESC NULLS LAST
    LIMIT 1;
"""
df_q3 = pd.read_sql(sql_q3, conexao)
df_q3

In [ ]:
linha = df_q3.iloc[0]
print(f"Viagem {linha['id_viagem']} ({linha['nome_orgao_superior']}) durou "
      f"{linha['duracao_dias']} dias, de {linha['data_inicio']} a {linha['data_fim']}, "
      f"com custo total de R$ {linha['valor_total']:.2f}.")

## 3. Camada Gold -- tabela agregada + view

Agora construímos a camada Gold: uma tabela que agrega, por **órgão pagador**,
o total pago, a média por pagamento e a duração média das viagens associadas
(usando `JOIN` entre `silver_pagamento` e `silver_viagem`, com `GROUP BY`).

Criamos a mesma agregação de duas formas:
- `gold_orgao_resumo` -- **tabela** materializada (uma "foto" dos dados no
  momento em que o pipeline rodou);
- `gold_orgao_resumo_view` -- **view**, que sempre reflete o estado atual das
  tabelas Silver.


In [ ]:
sql_gold_definicao = """
    SELECT
        p.nome_orgao_pagador,
        COUNT(DISTINCT p.id_viagem)   AS qtd_viagens,
        SUM(p.valor)                  AS valor_total_pago,
        ROUND(AVG(p.valor), 2)        AS valor_medio_pagamento,
        ROUND(AVG(v.duracao_dias), 1) AS duracao_media_dias
    FROM silver_pagamento p
    JOIN silver_viagem v ON v.id_viagem = p.id_viagem
    GROUP BY p.nome_orgao_pagador
"""

cursor = conexao.cursor()
cursor.execute("DROP TABLE IF EXISTS gold_orgao_resumo")
cursor.execute(f"CREATE TABLE gold_orgao_resumo AS {sql_gold_definicao}")
cursor.execute(f"CREATE OR REPLACE VIEW gold_orgao_resumo_view AS {sql_gold_definicao}")
conexao.commit()
cursor.close()
print("Tabela 'gold_orgao_resumo' e view 'gold_orgao_resumo_view' criadas.")

### Pergunta 7 -- Qual órgão pagou mais no total? (via camada Gold)

In [ ]:
df_gold_top = pd.read_sql(
    "SELECT * FROM gold_orgao_resumo_view ORDER BY valor_total_pago DESC LIMIT 5;",
    conexao,
)
df_gold_top

In [ ]:
plt.figure(figsize=(8, 5))
plt.barh(df_gold_top["nome_orgao_pagador"], df_gold_top["valor_total_pago"], color="#F18F01")
plt.xlabel("Valor total pago (R$)")
plt.title("Top 5 órgãos pagadores -- camada Gold")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 4. Perguntas de negócio 4 a 6

In [ ]:
sql_q4 = """
    SELECT tipo_pagamento,
           COUNT(*) AS qtd_pagamentos,
           ROUND(AVG(valor), 2) AS valor_medio
    FROM silver_pagamento
    GROUP BY tipo_pagamento
    ORDER BY valor_medio DESC;
"""
df_q4 = pd.read_sql(sql_q4, conexao)
df_q4

In [ ]:
plt.figure(figsize=(7, 4))
plt.bar(df_q4["tipo_pagamento"], df_q4["valor_medio"], color="#3B1F2B")
plt.ylabel("Valor médio (R$)")
plt.title("Valor médio por tipo de pagamento")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

### Pergunta 5 -- Qual o meio de transporte mais usado nos trechos?

In [ ]:
sql_q5 = """
    SELECT meio_transporte, COUNT(*) AS qtd_trechos
    FROM silver_trecho
    GROUP BY meio_transporte
    ORDER BY qtd_trechos DESC;
"""
df_q5 = pd.read_sql(sql_q5, conexao)
df_q5

In [ ]:
plt.figure(figsize=(7, 4))
plt.bar(df_q5["meio_transporte"], df_q5["qtd_trechos"], color="#0B6E4F")
plt.ylabel("Quantidade de trechos")
plt.title("Trechos por meio de transporte")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()


### Pergunta 6 -- Qual UF de destino aparece em mais trechos?

In [ ]:
sql_q6 = """
    SELECT destino_uf, COUNT(*) AS qtd_trechos
    FROM silver_trecho
    WHERE destino_uf IS NOT NULL
    GROUP BY destino_uf
    ORDER BY qtd_trechos DESC
    LIMIT 10;
"""
df_q6 = pd.read_sql(sql_q6, conexao)
df_q6


In [ ]:
plt.figure(figsize=(8, 5))
plt.barh(df_q6["destino_uf"], df_q6["qtd_trechos"], color="#6A4C93")
plt.xlabel("Quantidade de trechos")
plt.title("Top 10 UFs de destino mais frequentes nos trechos")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()
